# Monte Carlo Loss Sampling for ARR 2019

**Companion notebook to:** _Sampling ARR 2019 Loss Distributions in Python — A URBS Pre-Processor_

**R source:** Tony Ladson, [The Distribution of Losses](https://tonyladson.wordpress.com/2019/07/23/the-distribution-of-losses/)  
**Gist:** https://gist.github.com/TonyLadson/030860a88d3956627746e7acb99c3af9

---

## What this notebook does

Implements a Python sampler for drawing Initial Loss (IL) and Continuing Loss (CL) pairs from the
ARR 2019 empirical distributions (ARR Book 5, Table 5.3.13).

The method follows Nathan et al. (2003) and is described in ARR Book 5, Chapter 3.6.1:

1. The standardised IL and CL distributions from ARR Table 5.3.13 are treated as empirical CDFs
2. Random uniform samples are drawn and inverse-transformed through the empirical CDF
3. The standardised values are scaled by the median IL/CL for the project catchment

In R, `approxfun()` builds the inverse CDF. Here we use `scipy.interpolate.interp1d`.

**Key references:**
- ARR (2019), Book 5, Chapter 3.6.1 and Table 5.3.13
- Nathan, R.J. et al. (2003), IHWS Proceedings
- Ladson, A.R. (2019), https://tonyladson.wordpress.com/2019/07/23/the-distribution-of-losses/


---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.interpolate import interp1d

# Reproducibility
rng = np.random.default_rng(seed=42)

print('numpy:', np.__version__)
print('pandas:', pd.__version__)

---
## 2. ARR Table 5.3.13 — Standardised Loss Factors

Source: ARR 2019, Book 5, Table 5.3.13  
Original data: Hill et al. (2014), Project 6 Phase 4

The table gives standardised IL and CL at 11 percentile levels (0th to 100th).  
"Standardised" means divided by the median — so the 50th percentile is 1.0 for both IL and CL.

```r
# R reference (Ladson gist):
# loss_std <- structure(list(
#   Percentile = c(100, 90, 80, 70, 60, 50, 40, 30, 20, 10, 0),
#   IL = c(0.14, 0.39, 0.53, 0.68, 0.85, 1, 1.2, 1.4, 1.71, 2.26, 3.19),
#   CL = c(0.15, 0.35, 0.48, 0.61, 0.79, 1, 1.24, 1.5, 1.88, 2.48, 3.85),
#   prob = c(0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1)
# ))
# Note: Ladson re-orders by prob = (100 - Percentile) / 100 ascending
```


In [ ]:
# ARR Table 5.3.13 — ordered from 0th to 100th percentile (prob 0 to 1)
# prob = fraction of time loss is EXCEEDED (i.e. 0 = always exceeded, 1 = never exceeded)
# Rearranged to ascending probability for interp1d

loss_std = pd.DataFrame({
    'percentile': [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    'prob':       [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    'IL_std':     [0.14, 0.39, 0.53, 0.68, 0.85, 1.00, 1.20, 1.40, 1.71, 2.26, 3.19],
    'CL_std':     [0.15, 0.35, 0.48, 0.61, 0.79, 1.00, 1.24, 1.50, 1.88, 2.48, 3.85],
})

print(loss_std.to_string(index=False))

---
## 3. Build the Inverse CDF Functions

In R, `approxfun(x = prob, y = IL_std)` builds a linear interpolation function.  
Here we use `scipy.interpolate.interp1d` with `kind='linear'` — equivalent behaviour.

```r
# R reference:
# IL_f <- approxfun(x = 100 - loss_std$Percentile, y = loss_std$IL)
# IL_f(100 * runif(10))   # draw 10 random standardised IL values
#
# CL_f <- approxfun(x = 100 - loss_std$Percentile, y = loss_std$CL)
# CL_f(100 * runif(10))   # draw 10 random standardised CL values
```


In [ ]:
# Build inverse CDF (quantile function) for IL and CL
# interp1d maps from probability [0,1] to standardised loss value

IL_icdf = interp1d(loss_std['prob'], loss_std['IL_std'],
                   kind='linear', bounds_error=True)

CL_icdf = interp1d(loss_std['prob'], loss_std['CL_std'],
                   kind='linear', bounds_error=True)

# Quick check: 50th percentile should return 1.0 for both
print(f'IL at 50th percentile: {IL_icdf(0.5):.3f}  (expected 1.000)')
print(f'CL at 50th percentile: {CL_icdf(0.5):.3f}  (expected 1.000)')

---
## 4. Visualise the Empirical CDFs

Reproduce Figure 1 from the Ladson blog post — the empirical CDF of standardised IL and CL.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, col, label in zip(axes,
                           ['IL_std', 'CL_std'],
                           ['Standardised Initial Loss', 'Standardised Continuing Loss']):
    ax.plot(loss_std[col], loss_std['prob'], 'o-', color='steelblue', lw=1.5, ms=5)
    ax.axhline(0.5, color='grey', lw=0.8, linestyle='--', label='50th percentile')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Cumulative probability', fontsize=11)
    ax.set_xlim(0, None)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle('ARR 2019 Table 5.3.13 — Standardised Loss Distributions', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-04_mc-loss-cdf.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Generate Random Standardised Loss Samples

Draw N uniform random numbers in [0, 1] and pass through the inverse CDF.
This is the Monte Carlo sampling step.

```r
# R reference:
# ILs <- tibble(IL = IL_f(100 * runif(10000)))
# CLs <- tibble(CL = CL_f(100 * runif(10000)))
# mean(ILs$IL)    # ~1.15
# median(ILs$IL)  # ~1.00
```


In [ ]:
N = 10_000

# Draw N uniform random probabilities
u_IL = rng.uniform(0, 1, N)
u_CL = rng.uniform(0, 1, N)

# Map through inverse CDF to get standardised loss values
IL_samples = IL_icdf(u_IL)
CL_samples = CL_icdf(u_CL)

print(f'IL — mean: {IL_samples.mean():.3f}, median: {np.median(IL_samples):.3f}')
print(f'CL — mean: {CL_samples.mean():.3f}, median: {np.median(CL_samples):.3f}')
print(f'\nExpected from R: IL mean ~1.15, median ~1.00')
print(f'               CL mean ~1.24, median ~1.00')

---
## 6. Visualise the Sampled Distributions

Reproduce Figures 3 and 4 from the Ladson blog post — histograms of sampled IL and CL.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, samples, label in zip(axes,
                               [IL_samples, CL_samples],
                               ['Standardised Initial Loss', 'Standardised Continuing Loss']):
    ax.hist(samples, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(np.median(samples), color='orange', lw=1.5, linestyle='--',
               label=f'Median = {np.median(samples):.2f}')
    ax.axvline(samples.mean(), color='red', lw=1.5, linestyle=':',
               label=f'Mean = {samples.mean():.2f}')
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle(f'ARR 2019 Standardised Loss Distributions — N={N:,} samples', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-04_mc-loss-histograms.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Scale to Project Median IL/CL

The samples above are standardised (divided by the median).  
To get actual IL/CL values for a project catchment, multiply by the project median.

The project median IL and CL come from:
- ARR Data Hub (for your specific lat/lon, duration and AEP), or
- Regional loss tables (e.g. Ladson 2021 Victorian loss regions)


In [ ]:
# TODO: Replace with project-specific median IL and CL from the ARR Data Hub
# For demonstration, using typical SE Queensland values for a 1% AEP, 6-hour event

MEDIAN_IL_mm = 20.0   # mm  — replace with ARR Data Hub value for your catchment
MEDIAN_CL_mm_hr = 2.5 # mm/hr — replace with ARR Data Hub value for your catchment

# Scale standardised samples to actual values
IL_actual = IL_samples * MEDIAN_IL_mm
CL_actual = CL_samples * MEDIAN_CL_mm_hr

# Clip negative values to zero (physically impossible)
IL_actual = np.maximum(IL_actual, 0)
CL_actual = np.maximum(CL_actual, 0)

print(f'Scaled IL — mean: {IL_actual.mean():.1f} mm, median: {np.median(IL_actual):.1f} mm')
print(f'Scaled CL — mean: {CL_actual.mean():.2f} mm/hr, median: {np.median(CL_actual):.2f} mm/hr')

---
## 8. Convergence Check — How Many Ensemble Members?

How many IL/CL pairs do you need before the sample statistics stabilise?  
Plot the running mean of IL and CL as N increases.


In [ ]:
# Running mean of IL and CL samples
ns = np.arange(1, N + 1)
running_mean_IL = np.cumsum(IL_actual) / ns
running_mean_CL = np.cumsum(CL_actual) / ns

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, running_mean, median_val, label, unit in zip(
        axes,
        [running_mean_IL, running_mean_CL],
        [IL_actual.mean(), CL_actual.mean()],
        ['Initial Loss', 'Continuing Loss'],
        ['mm', 'mm/hr']):
    ax.plot(ns, running_mean, lw=0.8, color='steelblue')
    ax.axhline(median_val, color='red', lw=1, linestyle='--',
               label=f'Full-sample mean = {median_val:.2f} {unit}')
    ax.axvline(100, color='grey', lw=0.8, linestyle=':', label='N=100')
    ax.set_xlabel('Number of ensemble members (N)', fontsize=10)
    ax.set_ylabel(f'Running mean {label} ({unit})', fontsize=10)
    ax.set_xscale('log')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Convergence of Monte Carlo loss statistics with ensemble size', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-04_mc-loss-convergence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Export IL/CL Pairs — URBS Batch Input Format

Write a CSV with N rows, one per ensemble member.  
Format is designed to feed directly into a URBS batch pre-processor.


In [ ]:
def export_il_cl_samples(IL_actual, CL_actual, n_members, filepath):
    """
    Export N IL/CL pairs to CSV for URBS batch input.
    
    Parameters
    ----------
    IL_actual : np.ndarray  — actual IL values (mm)
    CL_actual : np.ndarray  — actual CL values (mm/hr)
    n_members : int         — number of ensemble members to export
    filepath  : str         — output CSV path
    """
    df = pd.DataFrame({
        'member': np.arange(1, n_members + 1),
        'IL_mm':  IL_actual[:n_members].round(1),
        'CL_mm_hr': CL_actual[:n_members].round(2),
    })
    df.to_csv(filepath, index=False)
    print(f'Written {n_members} IL/CL pairs to {filepath}')
    return df


# Export three ensemble sizes
for n in [100, 500, 1000]:
    export_il_cl_samples(
        IL_actual, CL_actual,
        n_members=n,
        filepath=f'data/il_cl_samples_N{n:04d}.csv'
    )

# Preview the first few rows of the 100-member file
pd.read_csv('data/il_cl_samples_N0100.csv').head(10)

---
## 10. TODO — LYR RORB Application

This section is a placeholder for the Lower Yarra River (LYR) RORB Monte Carlo application.  
Lindsay to populate with project-specific data.


In [ ]:
# TODO: LYR RORB Monte Carlo application
# ----------------------------------------
# 1. Load LYR median IL and CL from ARR Data Hub (or from project report)
# 2. Replace MEDIAN_IL_mm and MEDIAN_CL_mm_hr above with LYR values
# 3. Export il_cl_samples_LYR_N100.csv for input to RORB batch runs
# 4. Document the sensitivity of peak flow quantiles to N (100 vs 500 vs 1000)

# MEDIAN_IL_mm = ???    # from ARR Data Hub for LYR design event
# MEDIAN_CL_mm_hr = ???  # from ARR Data Hub for LYR design event

print('LYR application — to be completed')

---
## References

- Ball, J. et al. (2019). *Australian Rainfall and Runoff*. Book 5, Chapter 3 and Table 5.3.13.
- Hill, P., Graszkiewicz, Z., Taylor, M., Nathan, R. (2014). Project 6: Loss models for catchment simulation: Phase 4 analysis of rural catchments. ARR Revision Projects.
- Nathan, R.J., Weinmann, P.E. and Hill, P.I. (2003). Use of a Monte-Carlo Simulation to estimate the Expected Probability of large to extreme floods. *28th IHWS*, Wollongong.
- Ladson, A.R. (2019). The distribution of losses. https://tonyladson.wordpress.com/2019/07/23/the-distribution-of-losses/
